# 02 - ACE Training
Train the localized ACE model on the dataset and log metrics.


In [1]:
import sys
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.loader import DataLoader
import pandas as pd

sys.path.append("..")
from src.dataset import MaterialsProjectDataset
from src.models.ace_wrapper import ACEWrapper
from src.trainer import BenchmarkTrainer


In [2]:
# Load Data
train_ds = MaterialsProjectDataset("../data/train.extxyz", cutoff=5.0)
val_ds = MaterialsProjectDataset("../data/val.extxyz", cutoff=5.0)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)


Pre-computing graphs for 1000 structures (cutoff=5.0 Å)...


Building graphs:   0%|          | 0/1000 [00:00<?, ?it/s]

  Done. Dataset ready (1000 graphs).
Pre-computing graphs for 200 structures (cutoff=5.0 Å)...


Building graphs:   0%|          | 0/200 [00:00<?, ?it/s]

  Done. Dataset ready (200 graphs).


In [3]:
# Initialize Model with identical hyperparameters
model = ACEWrapper(
    num_radial=8, 
    l_max=2, 
    r_cut=5.0, 
    hidden_dim=32
)

optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

trainer = BenchmarkTrainer(
    model=model,
    optimizer=optimizer,
    scheduler=scheduler,
    train_loader=train_loader,
    val_loader=val_loader,
    device="cuda" if torch.cuda.is_available() else "cpu",
    energy_weight=1.0,
    force_weight=100.0
)


C:\Users\Prabhat\AppData\Local\Programs\Python\Python314\Lib\ast.py:506: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  return visitor(node)
C:\Users\Prabhat\AppData\Local\Programs\Python\Python314\Lib\ast.py:506: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  return visitor(node)


In [4]:
# Train
metrics_df = trainer.train(max_epochs=50, patience=10)
metrics_df.to_csv("../data/ace_metrics.csv", index=False)
metrics_df.head()


Epoch 000 | Time: 4.70s | Train E MAE: 14653.58 meV/atom | Train F MAE: 99.95 meV/Å | Val E MAE: 14645.19 meV/atom | Val F MAE: 164.77 meV/Å
Epoch 001 | Time: 3.44s | Train E MAE: 12657.36 meV/atom | Train F MAE: 360.83 meV/Å | Val E MAE: 12455.19 meV/atom | Val F MAE: 469.34 meV/Å
Epoch 002 | Time: 3.27s | Train E MAE: 11407.48 meV/atom | Train F MAE: 623.97 meV/Å | Val E MAE: 11974.11 meV/atom | Val F MAE: 432.55 meV/Å
Epoch 003 | Time: 3.26s | Train E MAE: 11099.09 meV/atom | Train F MAE: 469.81 meV/Å | Val E MAE: 11711.17 meV/atom | Val F MAE: 354.19 meV/Å
Epoch 004 | Time: 3.22s | Train E MAE: 10987.99 meV/atom | Train F MAE: 414.99 meV/Å | Val E MAE: 11458.13 meV/atom | Val F MAE: 384.21 meV/Å
Epoch 005 | Time: 3.49s | Train E MAE: 10934.02 meV/atom | Train F MAE: 516.65 meV/Å | Val E MAE: 11204.98 meV/atom | Val F MAE: 508.57 meV/Å
Epoch 006 | Time: 3.66s | Train E MAE: 10385.31 meV/atom | Train F MAE: 718.59 meV/Å | Val E MAE: 10948.41 meV/atom | Val F MAE: 717.94 meV/Å
Epoch 0

,epoch,loss,e_mae,f_mae,time,val_loss,val_e_mae,val_f_mae
0,0,116353.668213,14653.581679,99.948748,4.703369,82168.744420,14645.189966,164.768439
1,1,75063.954010,12657.361656,360.827832,3.436108,53516.303013,12455.187525,469.342755
2,2,56620.136963,11407.482162,623.974965,3.265224,49528.899972,11974.113328,432.552254
3,3,51828.922302,11099.094629,469.809812,3.262719,47180.778181,11711.166246,354.189095
4,4,48767.905029,10987.994045,414.994586,3.216014,44606.977400,11458.133493,384.207872
